# Hindi voice-clone detection: dataset and baseline

Run the cells in order, in one Colab session. Each step prints a summary when it finishes.

Before you start:

1. Create a free account at huggingface.co.
2. Accept the terms on the [Kathbath dataset page](https://huggingface.co/datasets/ai4bharat/Kathbath).
3. Create a "Read" access token under Settings → Access Tokens.

No audio or tokens are stored in this notebook.

## Log in to Hugging Face

Paste your token when asked. Do not type it into a cell.

In [ ]:
from huggingface_hub import login
login()

## Step 1: download a sample of real and fake Hindi clips

Downloads 14 large files one at a time (about 8 GB, each deleted after use) and keeps 3,000 fake clips from IndicSynth and 3,000 real clips from Kathbath, spread across many speakers. Safe to re-run: finished files are skipped.

In [ ]:
import os, io, glob
import numpy as np, pandas as pd, pyarrow.parquet as pq, soundfile as sf
from huggingface_hub import hf_hub_download

OUT = "/content/data/hindi"     # where the clips and the list of clips end up
TMP = "/content/hf_tmp"         # big downloads go here and are deleted after use

FAKE_REPO, REAL_REPO = "vdivyasharma/IndicSynth", "ai4bharat/Kathbath"
# 8 of the 107 fake files: 4 from the freevc24 half, 4 from the xtts_v2 half
FAKE_FILES = [f"Hindi/train-{i:05d}-of-00107.parquet" for i in (4, 17, 30, 43, 62, 75, 88, 101)]
# 6 of the 32 real files, spread out
REAL_FILES = [f"hindi/train-{i:05d}-of-00032.parquet" for i in (2, 8, 13, 19, 24, 29)]
PER_FILE = {"fake": 375, "real": 500}   # 8 x 375 = 3000 fake, 6 x 500 = 3000 real


def kind(b):
    """Work out the audio format from the first few bytes."""
    if b[:4] == b"RIFF": return "wav"
    if b[4:8] == b"ftyp": return "m4a"
    if b[:4] == b"fLaC": return "flac"
    if b[:4] == b"OggS": return "ogg"
    if b[:3] == b"ID3" or b[:2] in (b"\xff\xfb", b"\xff\xf3"): return "mp3"
    return "bin"


def grab(repo, fname, label):
    tag = f"{label}_{fname.split('-')[1]}"            # e.g. fake_00004
    part = f"{OUT}/parts/{tag}.csv"
    if os.path.exists(part):
        print("already done:", tag)
        return
    print("downloading", fname, "...")
    local = hf_hub_download(repo, fname, repo_type="dataset", local_dir=TMP)
    pf = pq.ParquetFile(local)
    n = pf.metadata.num_rows
    want = np.unique(np.linspace(0, n - 1, min(PER_FILE[label], n)).astype(int))
    acol = "audio" if label == "fake" else "audio_filepath"
    rows, start, skipped = [], 0, 0
    for g in range(pf.num_row_groups):
        size = pf.metadata.row_group(g).num_rows
        pick = want[(want >= start) & (want < start + size)] - start
        if len(pick):
            for i, r in zip(pick, pf.read_row_group(g).take(pick).to_pylist()):
                b = (r[acol] or {}).get("bytes")
                if not b:
                    skipped += 1
                    continue
                fmt = kind(b)
                path = f"{OUT}/{label}/{tag}_{start + i:05d}.{fmt}"
                with open(path, "wb") as f:
                    f.write(b)
                try:
                    info = sf.info(io.BytesIO(b))
                    sr, dur = info.samplerate, info.frames / info.samplerate
                except Exception:
                    sr, dur = None, r.get("duration")
                if label == "fake":
                    rows.append(dict(path=path, label="fake", tool=r["Generative Model"],
                                     speaker_id=r["Target Speaker ID"], source_speaker_id=r["Source Speaker_ID"],
                                     gender=r["Gender"], ref=r["Target Reference Audio"],
                                     text=r["TTS Transcript"], fmt=fmt, sr=sr, duration=dur))
                else:
                    rows.append(dict(path=path, label="real", tool="real",
                                     speaker_id=r["speaker_id"], source_speaker_id=None,
                                     gender=r["gender"], ref=r["fname"],
                                     text=r["text"], fmt=fmt, sr=sr, duration=dur))
        start += size
    pd.DataFrame(rows).to_csv(part, index=False)
    os.remove(local)
    print(f"  kept {len(rows)} of {n} clips" + (f" ({skipped} had no audio)" if skipped else ""))


for d in ("fake", "real", "parts"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
for f in FAKE_FILES:
    grab(FAKE_REPO, f, "fake")
for f in REAL_FILES:
    grab(REAL_REPO, f, "real")

meta = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f"{OUT}/parts/*.csv"))], ignore_index=True)
meta["gender"] = meta["gender"].str.lower()
meta.to_csv(f"{OUT}/meta.csv", index=False)

print("\n=== SUMMARY ===")
print(meta.groupby("tool").agg(clips=("path", "size"), speakers=("speaker_id", "nunique"),
                               avg_seconds=("duration", "mean"), shortest=("duration", "min"),
                               longest=("duration", "max")).round(2))
print("\nFormat and sample rate:")
print(meta.groupby(["tool", "fmt", "sr"], dropna=False).size())
print("\nGender:")
print(meta.groupby(["tool", "gender"]).size())
fake_spk = set(meta[meta.label == "fake"].speaker_id)
real_spk = set(meta[meta.label == "real"].speaker_id)
print(f"\nSpeakers: {len(fake_spk)} imitated, {len(real_spk)} real, {len(fake_spk & real_spk)} in both")
print("Example real file name:", meta[meta.label == "real"].ref.iloc[0])
print("Example fake reference:", meta[meta.label == "fake"].ref.iloc[0])

## Step 2: more `xtts_v2` voices, one audio format, split by speaker

Adds 12 more `xtts_v2` files (about 7 GB) so that tool has more than a handful of voices, converts every clip to 16 kHz mono WAV, and puts each speaker in exactly one of train, validation and test.

In [ ]:
import os, glob
from math import gcd
import numpy as np, pandas as pd, soundfile as sf
from scipy.signal import resample_poly
from tqdm.auto import tqdm

OUT16 = "/content/data/hindi16k"

# --- A. More xtts_v2 voices: 12 more files, 125 clips from each ---------------
PER_FILE["fake"] = 125
for i in (57, 60, 65, 68, 71, 78, 81, 84, 91, 94, 97, 104):
    grab(FAKE_REPO, f"Hindi/train-{i:05d}-of-00107.parquet", "fake")

meta = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f"{OUT}/parts/*.csv"))], ignore_index=True)
meta["gender"] = meta["gender"].str.lower()

# --- B. Keep about 1500 xtts_v2 clips, the same number from every voice -------
x = meta[meta.tool == "xtts_v2"]
per_voice = max(1, 1500 // x.speaker_id.nunique())
x = x.sample(frac=1, random_state=0).groupby("speaker_id").head(per_voice)
meta = pd.concat([meta[meta.tool != "xtts_v2"], x], ignore_index=True)

# --- C. Split by speaker: each person is in train, val or test, never two -----
who = meta.groupby("speaker_id").agg(gender=("gender", "first"),
                                     has_xtts=("tool", lambda t: (t == "xtts_v2").any()))
rng = np.random.default_rng(0)
split = {}
for _, group in who.groupby(["has_xtts", "gender"]):
    ids = rng.permutation(group.index.to_numpy())
    k = max(1, round(0.15 * len(ids))) if len(ids) >= 3 else 0
    for j, s in enumerate(ids):
        split[s] = "test" if j < k else "val" if j < 2 * k else "train"
meta["split"] = meta.speaker_id.map(split)
assert meta.groupby("speaker_id").split.nunique().max() == 1

# --- D. Convert every clip to the same format: 16 kHz, mono, WAV --------------
for d in ("fake", "real"):
    os.makedirs(f"{OUT16}/{d}", exist_ok=True)

wavs, loud, secs = [], [], []
for src, label in tqdm(list(zip(meta.path, meta.label)), desc="converting"):
    dst = f"{OUT16}/{label}/{os.path.splitext(os.path.basename(src))[0]}.wav"
    a, sr = sf.read(src, dtype="float32", always_2d=True)
    a = a.mean(axis=1)
    if sr != 16000:
        g = gcd(sr, 16000)
        a = resample_poly(a, 16000 // g, sr // g)
    a = np.clip(a, -1.0, 1.0).astype("float32")
    sf.write(dst, a, 16000, subtype="PCM_16")
    wavs.append(dst)
    loud.append(20 * np.log10(np.sqrt(np.mean(a ** 2)) + 1e-9))
    secs.append(len(a) / 16000)

meta["orig_path"], meta["orig_sr"] = meta["path"], meta["sr"]
meta["path"], meta["sr"], meta["duration"], meta["loudness_db"] = wavs, 16000, secs, loud
meta = meta.drop(columns=["fmt"])
meta.to_csv(f"{OUT16}/meta.csv", index=False)

print("\n=== SUMMARY ===")
print("Clips:")
print(meta.pivot_table(index="tool", columns="split", values="path", aggfunc="size", fill_value=0)[["train", "val", "test"]])
print("\nSpeakers:")
print(meta.pivot_table(index="tool", columns="split", values="speaker_id", aggfunc="nunique", fill_value=0)[["train", "val", "test"]])
print("\nLoudness (dB) and length (seconds):")
print(meta.groupby("tool").agg(loudness=("loudness_db", "mean"), loudness_spread=("loudness_db", "std"),
                               avg_seconds=("duration", "mean")).round(1))
print("\nGender:")
print(meta.pivot_table(index="tool", columns="gender", values="path", aggfunc="size", fill_value=0))
print(f"\n{len(meta)} clips saved in {OUT16}")

## Optional: save the finished dataset to Google Drive

Colab deletes its files when the session ends. This keeps a copy (about 1 GB).

In [ ]:
from google.colab import drive
import shutil, os

drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/voice-clone-detector", exist_ok=True)
shutil.make_archive("/content/hindi16k", "zip", "/content/data/hindi16k")
shutil.copy("/content/hindi16k.zip", "/content/drive/MyDrive/voice-clone-detector/hindi16k.zip")
print("Saved to Google Drive")

## Step 3: baseline, AASIST on the clean Hindi test clips

Downloads AASIST (trained on English ASVspoof 2019 data), sets every clip to the same loudness and length, and reports the Equal Error Rate: 0% is perfect, 50% is a coin flip. About 10 minutes on a CPU.

In [ ]:
import os, json, subprocess, importlib.util
import numpy as np, pandas as pd, soundfile as sf, torch
from sklearn.metrics import roc_curve
from tqdm.auto import tqdm

OUT16 = "/content/data/hindi16k"
AASIST_DIR = "/content/aasist"
DRIVE = "/content/drive/MyDrive/voice-clone-detector"
RESULTS = f"{DRIVE}/results" if os.path.isdir(DRIVE) else "/content/results"

# --- 1. Get AASIST, a standard detector trained on English research data -----
if not os.path.exists(AASIST_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/clovaai/aasist", AASIST_DIR], check=True)
spec = importlib.util.spec_from_file_location("aasist_model", f"{AASIST_DIR}/models/AASIST.py")
aasist = importlib.util.module_from_spec(spec)
spec.loader.exec_module(aasist)

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = json.load(open(f"{AASIST_DIR}/config/AASIST.conf"))["model_config"]
model = aasist.Model(cfg).to(device)
model.load_state_dict(torch.load(f"{AASIST_DIR}/models/weights/AASIST.pth", map_location=device))
model.eval()
print("AASIST loaded on", device)

# --- 2. One way of loading a clip, used for every test from now on -----------
CLIP_LEN = 64600      # about 4 seconds, the length AASIST was trained on
TARGET_DB = -23.0     # every clip is set to this loudness


def load_clip(path):
    a, sr = sf.read(path, dtype="float32")
    assert sr == 16000, f"{path} is {sr} Hz"
    a = a - a.mean()
    a = a * (10 ** (TARGET_DB / 20) / (np.sqrt(np.mean(a ** 2)) + 1e-9))
    if len(a) < CLIP_LEN:
        a = np.tile(a, CLIP_LEN // len(a) + 1)
    return a[:CLIP_LEN]


def score_clips(paths, batch=16):
    """Returns one number per clip. Higher means the detector thinks it is real."""
    realness, says_real = [], []
    for i in tqdm(range(0, len(paths), batch), desc="scoring"):
        x = torch.from_numpy(np.stack([load_clip(p) for p in paths[i:i + batch]])).to(device)
        with torch.no_grad():
            _, out = model(x)
        realness += out[:, 1].cpu().tolist()
        says_real += (out[:, 1] > out[:, 0]).cpu().tolist()
    return np.array(realness), np.array(says_real)


def eer(is_real, score):
    """Equal Error Rate in percent. 0 is perfect, 50 is a coin flip."""
    fpr, tpr, _ = roc_curve(is_real, score)
    i = np.nanargmin(np.abs((1 - tpr) - fpr))
    return 100 * (fpr[i] + 1 - tpr[i]) / 2


def eer_range(is_real, score, n=500, seed=0):
    """Rough 95% range for the EER, from re-drawing the test clips many times."""
    rng, vals = np.random.default_rng(seed), []
    for _ in range(n):
        j = rng.integers(0, len(score), len(score))
        if 0 < is_real[j].sum() < len(j):
            vals.append(eer(is_real[j], score[j]))
    return np.percentile(vals, [2.5, 97.5])


# --- 3. Score the clean test clips -------------------------------------------
meta = pd.read_csv(f"{OUT16}/meta.csv")
test = meta[meta.split == "test"].reset_index(drop=True)
test["realness"], test["says_real"] = score_clips(list(test.path))
test["is_real"] = (test.label == "real").astype(int)

os.makedirs(RESULTS, exist_ok=True)
test.to_csv(f"{RESULTS}/baseline_aasist_clean_hindi.csv", index=False)

print("\n=== BASELINE: AASIST on clean Hindi test clips ===")
real = test[test.label == "real"]
rows = [("all fakes", test)] + [(t, pd.concat([real, test[test.tool == t]])) for t in ("freevc24", "xtts_v2")]
for name, d in rows:
    y, s = d.is_real.to_numpy(), d.realness.to_numpy()
    lo, hi = eer_range(y, s)
    print(f"real vs {name:<10} EER {eer(y, s):5.1f}%   (likely range {lo:.1f} to {hi:.1f})   "
          f"{len(d) - y.sum()} fake, {y.sum()} real clips")

print("\nUsing the detector's own yes/no decision:")
print(f"  real clips wrongly called fake: {100 * (1 - real.says_real.mean()):.1f}%")
for t in ("freevc24", "xtts_v2"):
    print(f"  {t} fakes passed as real: {100 * test[test.tool == t].says_real.mean():.1f}%")
print("\nScores saved to", RESULTS)

## Step 4: check the test code on the data AASIST was built for

Scores 200 real and 200 fake clips from the ASVspoof 2019 LA test set with the same code. The published result for this model on the full set is 0.83%, so a figure near that means the Hindi numbers are not a bug. Downloads about 4.4 GB, deleted afterwards.

In [ ]:
import os
import numpy as np, pandas as pd, pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

CHK = "/content/data/asvspoof_check"
N_SCAN, N_EACH = 2400, 200      # look at 2400 clips spread over the set, keep 200 real + 200 fake

# --- 1. Get a sample of the English test set AASIST was built for ------------
if not os.path.exists(f"{CHK}/meta.csv"):
    os.makedirs(CHK, exist_ok=True)
    print("downloading the English test set (about 4.4 GB) ...")
    local = hf_hub_download("Bisher/ASVspoof_2019_LA", "data/test-00000-of-00001.parquet",
                            repo_type="dataset", local_dir="/content/hf_tmp")
    pf = pq.ParquetFile(local)
    n = pf.metadata.num_rows
    want = set(np.unique(np.linspace(0, n - 1, min(N_SCAN, n)).astype(int)).tolist())
    found, start = [], 0
    for batch in pf.iter_batches(batch_size=200):
        idx = [i for i in range(batch.num_rows) if start + i in want]
        if idx:
            found += batch.take(idx).to_pylist()
        start += batch.num_rows
    os.remove(local)

    df = pd.DataFrame(found)
    df["is_real"] = (df["key"] == 0).astype(int)        # in this copy: 0 = bonafide (real), 1 = spoof (fake)
    keep = pd.concat([d.sample(min(N_EACH, len(d)), random_state=0) for _, d in df.groupby("is_real")])
    rows = []
    for r in keep.to_dict("records"):
        path = f"{CHK}/{r['audio_file_name']}.flac"
        with open(path, "wb") as f:
            f.write(r["audio"]["bytes"])
        rows.append(dict(path=path, is_real=r["is_real"], system=r["system_id"]))
    pd.DataFrame(rows).to_csv(f"{CHK}/meta.csv", index=False)

# --- 2. Score it with exactly the same code used on the Hindi clips ----------
chk = pd.read_csv(f"{CHK}/meta.csv")
chk["realness"], chk["says_real"] = score_clips(list(chk.path))
y, s = chk.is_real.to_numpy(), chk.realness.to_numpy()
lo, hi = eer_range(y, s)

print("\n=== CHECK: AASIST on the English data it was built for ===")
print(f"EER {eer(y, s):.1f}%   (likely range {lo:.1f} to {hi:.1f})   {(y == 0).sum()} fake, {(y == 1).sum()} real clips")
print("Published result for this model on the full set: 0.83%")
print(f"  real clips wrongly called fake: {100 * (1 - chk[chk.is_real == 1].says_real.mean()):.1f}%")
print(f"  fakes passed as real: {100 * chk[chk.is_real == 0].says_real.mean():.1f}%")
print("  fake clips came from systems:", ", ".join(sorted(chk[chk.is_real == 0].system.astype(str).unique())))
print("  real clips are marked:", ", ".join(sorted(chk[chk.is_real == 1].system.astype(str).unique())))